Import necessary libraries first.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

Import the master dataframe with all coins and market features.

In [2]:
master_df = pd.read_csv("market-master-df.csv")

Ensure the data is in chronological order and seperated by symbol.

In [3]:
master_df['open_time'] = pd.to_datetime(master_df['open_time'])
master_df = master_df.sort_values(by=['open_time', 'symbol']).reset_index(drop=True)

In [4]:
print(master_df.head(10))

                  open_time     open     high      low    close      volume  \
0 2017-08-17 04:00:00+00:00  4261.48  4313.62  4261.32  4308.83   47.181009   
1 2017-08-17 04:00:00+00:00   301.13   302.57   298.00   301.61  125.668770   
2 2017-08-17 05:00:00+00:00  4308.83  4328.69  4291.37  4315.32   23.234916   
3 2017-08-17 05:00:00+00:00   301.61   303.28   300.00   303.10  377.672460   
4 2017-08-17 06:00:00+00:00  4330.29  4345.45  4309.37  4324.35    7.229691   
5 2017-08-17 06:00:00+00:00   302.40   304.44   301.90   302.68  303.866720   
6 2017-08-17 07:00:00+00:00  4316.62  4349.99  4287.41  4349.99    4.443249   
7 2017-08-17 07:00:00+00:00   302.68   307.96   302.60   307.96  754.745100   
8 2017-08-17 08:00:00+00:00  4333.32  4377.85  4333.32  4360.69    0.972807   
9 2017-08-17 08:00:00+00:00   307.95   309.97   307.00   308.62  150.750290   

    symbol  log_ret_1d  log_ret_5d  log_ret_21d  ...  log_volume  \
0  BTCUSDT         NaN         NaN          NaN  ...    3.8749

We need to make synthetic rows to ensure every time has an associated row for each coin, even if that coin doesn't exist yet. We do this because we need to ensure the shape of the data we feed to TS-JEPA retains a consistent shape for ingestion. However, these synthetic rows filled with NaNs for coins prior to their first hourly candle from the Binance API will ultimately be masked. For the exact procedure and explanation see [Joint-Embedding Predictive Learning of Latent Market States in U.S. Equities](https://openreview.net/pdf?id=BZfkxSasd3), Appendix B. Model Architecture, Handling missing assets subsection.

In [5]:
# Get a unique list of all timestamps and all 21 symbols
all_times = master_df['open_time'].unique()
all_symbols = master_df['symbol'].unique()

# Create a "Perfectly Square" MultiIndex
# This generates every possible combination of time and symbol
full_index = pd.MultiIndex.from_product(
    [all_times, all_symbols], 
    names=['open_time', 'symbol']
)

# Apply this index to your DataFrame
# Any coin that didn't exist at a specific timestamp will now get a synthetic row filled with NaNs
master_df = master_df.set_index(['open_time', 'symbol']).reindex(full_index).reset_index()

# Re-sort just to be absolutely sure of the chronological layout
master_df = master_df.sort_values(by=['open_time', 'symbol']).reset_index(drop=True)

Inspect new master dataframe with NaN samples.

In [6]:
print(master_df.head(22))

                   open_time    symbol     open     high      low    close  \
0  2017-08-17 04:00:00+00:00   ADAUSDT      NaN      NaN      NaN      NaN   
1  2017-08-17 04:00:00+00:00   BATUSDT      NaN      NaN      NaN      NaN   
2  2017-08-17 04:00:00+00:00   BCHUSDT      NaN      NaN      NaN      NaN   
3  2017-08-17 04:00:00+00:00   BNBUSDT      NaN      NaN      NaN      NaN   
4  2017-08-17 04:00:00+00:00   BTCUSDT  4261.48  4313.62  4261.32  4308.83   
5  2017-08-17 04:00:00+00:00  DASHUSDT      NaN      NaN      NaN      NaN   
6  2017-08-17 04:00:00+00:00   DGBUSDT      NaN      NaN      NaN      NaN   
7  2017-08-17 04:00:00+00:00  DOGEUSDT      NaN      NaN      NaN      NaN   
8  2017-08-17 04:00:00+00:00   ETCUSDT      NaN      NaN      NaN      NaN   
9  2017-08-17 04:00:00+00:00   ETHUSDT   301.13   302.57   298.00   301.61   
10 2017-08-17 04:00:00+00:00   FILUSDT      NaN      NaN      NaN      NaN   
11 2017-08-17 04:00:00+00:00  IOTAUSDT      NaN      NaN      Na

Isolate identifiers for z-score normalization.

In [7]:
metadata_cols = ['open_time', 'symbol']
feature_cols = [col for col in master_df.columns if col not in metadata_cols]

Now we define the train, evaluation, and test splits. The data is split as follows:

- `train` : first candle -> 2025-12-30
- `evaluation` : 2025-12-31 -> 2026-03-30
- `test` : 2026-03-31 -> 2026-07-15

NOTE: Data ends at 2026-07-15 00:00 UTC, where that time is the exact time of the last hourly candle in the test set.

This drops December 31st 2025, and March 31st 2026 from all splits to ensure we don't accidentally perform data leakage across splits.

In [8]:
train_mask = master_df['open_time'] < '2025-12-31 00:00:00'
eval_mask  = (master_df['open_time'] >= '2026-01-01 00:00:00') & (master_df['open_time'] < '2026-03-31 00:00:00')
test_mask  = (master_df['open_time'] >= '2026-04-01 00:00:00') & (master_df['open_time'] < '2026-07-16 00:00:00')

Now we create a validity mask for NaNs in our dataframe that we ultimately want TS-JEPA to ignore.

In [9]:
validity_df = master_df[['open_time', 'symbol']].copy()
validity_df[feature_cols] = (~master_df[feature_cols].isna()).astype(int)

In [10]:
print(validity_df.head(10))

                  open_time    symbol  open  high  low  close  volume  \
0 2017-08-17 04:00:00+00:00   ADAUSDT     0     0    0      0       0   
1 2017-08-17 04:00:00+00:00   BATUSDT     0     0    0      0       0   
2 2017-08-17 04:00:00+00:00   BCHUSDT     0     0    0      0       0   
3 2017-08-17 04:00:00+00:00   BNBUSDT     0     0    0      0       0   
4 2017-08-17 04:00:00+00:00   BTCUSDT     1     1    1      1       1   
5 2017-08-17 04:00:00+00:00  DASHUSDT     0     0    0      0       0   
6 2017-08-17 04:00:00+00:00   DGBUSDT     0     0    0      0       0   
7 2017-08-17 04:00:00+00:00  DOGEUSDT     0     0    0      0       0   
8 2017-08-17 04:00:00+00:00   ETCUSDT     0     0    0      0       0   
9 2017-08-17 04:00:00+00:00   ETHUSDT     1     1    1      1       1   

   log_ret_1d  log_ret_5d  log_ret_21d  ...  log_volume  log_dollar_volume  \
0           0           0            0  ...           0                  0   
1           0           0            0  

According to [Joint-Embedding Predictive Learning of Latent Market States in U.S. Equities](https://openreview.net/pdf?id=BZfkxSasd3), Appendix section A, subsection A.1, the train/validation/test splits should be normalized within respect to the training split ONLY. The following code initiates the scalar on the training set, and is then applied statically to the evaluation, and test sets.

In [11]:
# Initialize and Fit the Scaler ONLY on the Training Split
scaler = StandardScaler()
scaler.fit(master_df.loc[train_mask, feature_cols])

# Transform All Splits using the Training Distribution
master_df.loc[train_mask, feature_cols] = scaler.transform(master_df.loc[train_mask, feature_cols])
master_df.loc[eval_mask, feature_cols]  = scaler.transform(master_df.loc[eval_mask, feature_cols])
master_df.loc[test_mask, feature_cols]  = scaler.transform(master_df.loc[test_mask, feature_cols])

Now we fill the NaNs in the master df with 0s to ensure we don't feed NaNs into JEPA. However, our validity dataframe allows us to know when a zero is a "true zero" (one served by the Binance api) or a "fake zero" (one we made up to keep shape consistency). "fake zeros" will be effectively blocked out from the attention mask in JEPA.

In [12]:
master_df[feature_cols] = master_df[feature_cols].fillna(0.0)

Ensure there are zero NaNs left in our master df.

In [13]:
print(master_df.isna().sum())

open_time            0
symbol               0
open                 0
high                 0
low                  0
close                0
volume               0
log_ret_1d           0
log_ret_5d           0
log_ret_21d          0
log_ret_63d          0
log_ret_126d         0
intraday_ret         0
mom_6_1              0
hl_log_range         0
body_to_range        0
upper_shadow         0
lower_shadow         0
rvol_10              0
rvol_21              0
rvol_63              0
rvol_ratio           0
ewma_vol_hl10        0
ewma_vol_hl20        0
log_volume           0
log_dollar_volume    0
rel_dvol_21          0
dvol_z_21            0
amihud_illiq_1       0
amihud_illiq_21      0
mkt_log_ret_1        0
ret_vs_mkt_1         0
mkt_rvol_21          0
rel_vol              0
dtype: int64


No NaNs left. 🎉 Let's inspect the scaled values first before exporting.

In [14]:
print(master_df.head(22))

                   open_time    symbol      open      high       low  \
0  2017-08-17 04:00:00+00:00   ADAUSDT  0.000000  0.000000  0.000000   
1  2017-08-17 04:00:00+00:00   BATUSDT  0.000000  0.000000  0.000000   
2  2017-08-17 04:00:00+00:00   BCHUSDT  0.000000  0.000000  0.000000   
3  2017-08-17 04:00:00+00:00   BNBUSDT  0.000000  0.000000  0.000000   
4  2017-08-17 04:00:00+00:00   BTCUSDT  0.194346  0.197529  0.195915   
5  2017-08-17 04:00:00+00:00  DASHUSDT  0.000000  0.000000  0.000000   
6  2017-08-17 04:00:00+00:00   DGBUSDT  0.000000  0.000000  0.000000   
7  2017-08-17 04:00:00+00:00  DOGEUSDT  0.000000  0.000000  0.000000   
8  2017-08-17 04:00:00+00:00   ETCUSDT  0.000000  0.000000  0.000000   
9  2017-08-17 04:00:00+00:00   ETHUSDT -0.162694 -0.162778 -0.162747   
10 2017-08-17 04:00:00+00:00   FILUSDT  0.000000  0.000000  0.000000   
11 2017-08-17 04:00:00+00:00  IOTAUSDT  0.000000  0.000000  0.000000   
12 2017-08-17 04:00:00+00:00  LINKUSDT  0.000000  0.000000  0.00

Ensure scaling was done correctly.

In [15]:
# Filter to just the training split
train_df = master_df[train_mask]
train_validity = validity_df[train_mask]

# Pick a feature to test (e.g., 'open')
feature_to_test = 'open'

# Extract ONLY the rows where the validity mask is 1 (real data)
real_data = train_df.loc[train_validity[feature_to_test] == 1, feature_to_test]

# Check the statistics
print(f"Mean: {real_data.mean():.6f}")
print(f"Std Dev: {real_data.std():.6f}")

Mean: 0.000000
Std Dev: 1.000000


Now export the data. At this point we switch to parquet, because these will be the final artifacts fed into our models.

In [16]:
export_dir = "normalized-features"

# make directory if it doesn't exist
os.makedirs(export_dir, exist_ok=True)

# export training dataset and validity mask to parquet
train_df.to_parquet(os.path.join(export_dir, "train_df.parquet"), index=False)
train_validity.to_parquet(os.path.join(export_dir, "train_validity.parquet"), index=False)

# export evaluation dataset and validity mask to parquet
eval_df = master_df[eval_mask]
eval_validity = validity_df[eval_mask]

eval_df.to_parquet(os.path.join(export_dir, "eval_df.parquet"), index=False)
eval_validity.to_parquet(os.path.join(export_dir, "eval_validity.parquet"), index=False)

# export test dataset and validity mask to parquet
test_df = master_df[test_mask]
test_validity = validity_df[test_mask]

test_df.to_parquet(os.path.join(export_dir, "test_df.parquet"), index=False)
test_validity.to_parquet(os.path.join(export_dir, "test_validity.parquet"), index=False)